### Importing dependancies

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

### Visualization setting

In [3]:
PALETTE = {"No": "#2b5c8f", "Yes": "#d95f02"}

### Loading and preparing the dataset


In [4]:
fig_dir = "figures"
os.makedirs(fig_dir, exist_ok=True)
sns.set_theme(style="whitegrid", context="paper", font_scale=1.15)

df = pd.read_csv("telco.csv")
df["TotalCharges_num"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges_clean"] = df["TotalCharges_num"].fillna(0.0)

### Dataset Audit

In [5]:
print(f"Dataset Shape: {df.shape}")
print(f"TotalCharges Missing After Coercion: {df['TotalCharges_num'].isna().sum()}")
print("\nTarget Distribution (%):")
print((df["Churn"].value_counts(normalize=True) * 100).round(2))

Dataset Shape: (7043, 23)
TotalCharges Missing After Coercion: 11

Target Distribution (%):
Churn
No     73.46
Yes    26.54
Name: proportion, dtype: float64


### Target class balance figure

In [6]:
fig, ax = plt.subplots(figsize=(6.5, 4.2), dpi=300)
counts = df["Churn"].value_counts().reindex(["No", "Yes"])
bars = ax.bar(
    counts.index,
    counts.values,
    color=[PALETTE["No"], PALETTE["Yes"]],
    width=0.52,
    edgecolor="black",
    linewidth=0.8,
)
total = len(df)
for bar, count in zip(bars, counts.values):
    pct = count / total * 100
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        count + 110,
        f"{count:,}\n({pct:.2f}%)",
        ha="center",
        va="bottom",
        fontweight="bold",
        fontsize=10.5,
    )
ax.set_ylim(0, 6100)
ax.set_title(
    "Target Class Distribution: Customer Churn (N = 7,043)\nImbalance Ratio = 2.77 : 1",
    fontweight="bold",
    pad=12,
)
ax.set_xlabel("Churn Status", fontweight="bold")
ax.set_ylabel("Customer Count", fontweight="bold")
sns.despine(top=True, right=True)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "fig1_class_balance.png"), dpi=300)
plt.show()
plt.close()

C:\Users\LOQ\AppData\Local\Temp\ipykernel_22644\3592195398.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Churn Rates Across Key Categorical Drivers figure

Compare churn rates across three categorical features:
- **Contract:** month-to-month, one-year, and two-year contracts
- **Internet service:** fibre optic, DSL, and no internet service
- **Payment method:** the four available payment methods

In [7]:
fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.8), dpi=300, sharey=True)
cat_specs = [
    ("Contract", ["Month-to-month", "One year", "Two year"], "Contract Type"),
    ("InternetService", ["Fiber optic", "DSL", "No"], "Internet Service"),
    (
        "PaymentMethod",
        [
            "Electronic check",
            "Mailed check",
            "Bank transfer (automatic)",
            "Credit card (automatic)",
        ],
        "Payment Method",
    ),
]
for ax, (col, order, title) in zip(axes, cat_specs):
    rates = df.groupby(col)["Churn"].apply(lambda s: (s == "Yes").mean() * 100).reindex(order)
    totals = df[col].value_counts().reindex(order)
    short_labels = [lbl.replace(" (automatic)", "\n(auto)") for lbl in order]
    bars = ax.bar(
        range(len(order)),
        rates.values,
        color="#d95f02",
        alpha=0.88,
        width=0.55,
        edgecolor="black",
        linewidth=0.8,
    )
    ax.axhline(26.54, color="#2b5c8f", linestyle="--", linewidth=1.2, label="Global Churn (26.54%)")
    for bar, rate, n_cat in zip(bars, rates.values, totals.values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            rate + 1.2,
            f"{rate:.1f}%\n(n={n_cat:,})",
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight="bold",
        )
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels(
        short_labels,
        rotation=18 if col == "PaymentMethod" else 0,
        ha="center",
        fontsize=9.5,
    )
    ax.set_title(title, fontweight="bold")
    ax.set_ylim(0, 56)
    sns.despine(ax=ax, top=True, right=True)
axes[0].set_ylabel("Churn Rate (%)", fontweight="bold")
axes[2].legend(loc="upper right", frameon=True)
fig.suptitle(
    "Customer Churn Rate Across Primary Service & Billing Drivers",
    fontweight="bold",
    fontsize=13,
    y=1.02,
)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "fig2_categorical_churn_rates.png"), dpi=300, bbox_inches="tight")
plt.show()
plt.close()

C:\Users\LOQ\AppData\Local\Temp\ipykernel_22644\3957697638.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Tenure Distribution by Churn Status figure


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5), dpi=300)
for status in ["No", "Yes"]:
    subset = df[df["Churn"] == status]["tenure"]
    sns.kdeplot(
        subset,
        ax=ax,
        label=f"Churn = {status} (Median: {subset.median():.0f} mo)",
        color=PALETTE[status],
        fill=True,
        alpha=0.35,
        linewidth=2.0,
        clip=(0, 72),
    )
    ax.axvline(subset.median(), color=PALETTE[status], linestyle="--", linewidth=1.3)
ax.set_title("Customer Tenure Distribution by Churn Status (KDE)", fontweight="bold", pad=10)
ax.set_xlabel("Tenure (Months)", fontweight="bold")
ax.set_ylabel("Probability Density", fontweight="bold")
ax.set_xlim(0, 72)
ax.legend(frameon=True)
sns.despine(top=True, right=True)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "fig3_tenure_kde.png"), dpi=300)
plt.show()
plt.close()

### Monthly Charges Distribution by Churn Status figure


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5), dpi=300)
for status in ["No", "Yes"]:
    subset = df[df["Churn"] == status]["MonthlyCharges"]
    sns.kdeplot(
        subset,
        ax=ax,
        label=f"Churn = {status} (Median: ${subset.median():.2f})",
        color=PALETTE[status],
        fill=True,
        alpha=0.35,
        linewidth=2.0,
        clip=(15, 120),
    )
    ax.axvline(subset.median(), color=PALETTE[status], linestyle="--", linewidth=1.3)
ax.set_title("Monthly Charges Distribution by Churn Status (KDE)", fontweight="bold", pad=10)
ax.set_xlabel("Monthly Charges ($ USD)", fontweight="bold")
ax.set_ylabel("Probability Density", fontweight="bold")
ax.set_xlim(15, 120)
ax.legend(frameon=True)
sns.despine(top=True, right=True)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "fig4_monthly_charges_kde.png"), dpi=300)
plt.show()
plt.close()

### Pearson Correlation Matrix figure

In [ ]:
corr_df = df[["tenure", "MonthlyCharges", "TotalCharges_clean", "SeniorCitizen"]].copy()
corr_df["Churn_Binary"] = df["Churn"].map({"No": 0, "Yes": 1})
corr_df.columns = ["Tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen", "Churn (0/1)"]
corr_mat = corr_df.corr()

fig, ax = plt.subplots(figsize=(6.8, 5.4), dpi=300)
sns.heatmap(
    corr_mat,
    annot=True,
    fmt=".3f",
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    linewidths=0.8,
    cbar_kws={"shrink": 0.82},
    ax=ax,
)
ax.set_title("Pearson Correlation Matrix (Numeric Features & Target)", fontweight="bold", pad=12)
plt.xticks(rotation=25, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "fig5_correlation_heatmap.png"), dpi=300)
plt.show()
plt.close()